In [1]:
# STEP 1: Install dependencies
!pip install pandas scikit-learn fuzzywuzzy python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 23.5 MB/s eta 0:00:00


In [2]:
# STEP 2: Import libraries
import pandas as pd
from fuzzywuzzy import fuzz
from sklearn.impute import SimpleImputer

In [4]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

print(df.head())


   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [5]:
# STEP 4: Enrichment
df['full_name'] = df['who'].astype(str) + "_" + df['class'].astype(str)

In [6]:
# STEP 5: Handle missing values
imputer = SimpleImputer(strategy='most_frequent')
df[['embark_town']] = imputer.fit_transform(df[['embark_town']])

In [7]:
# STEP 6: Entity Resolution (fuzzy matching)
def find_duplicates(df):
    duplicates = []
    names = df['full_name'].astype(str).tolist()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            score = fuzz.ratio(names[i], names[j])
            if score > 90:
                duplicates.append((names[i], names[j], score))
    return duplicates

In [8]:
duplicates = find_duplicates(df)
print("\nSample duplicates:", duplicates[:5])


Sample duplicates: [('man_Third', 'man_Third', 100), ('man_Third', 'man_Third', 100), ('man_Third', 'man_Third', 100), ('man_Third', 'man_Third', 100), ('man_Third', 'man_Third', 100)]


In [9]:
# STEP 7: Save
df.to_csv("enriched_data.csv", index=False)